# Setup

In [1]:
master = "spark://spark-master:7077"
dataFolder = "/data/" if master.startswith("spark://") else "../data/"

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, LongType
from pyspark.sql import Window

spark_builder = (SparkSession.builder
         .master(master)
         .appName("spark-profiling-exercises")
         .config("spark.driver.memory", "2g")
         .config("spark.executor.memory", "1536m")   # stay under the worker's --memory 2G cap
         .config("spark.ui.showConsoleProgress", "false"))

# if master.startswith("spark://"):
#     spark_builder = (spark_builder
#                       .config("spark.driver.host", "localhost")
#                       .config("spark.driver.bindAddress", "0.0.0.0"))

spark = spark_builder.getOrCreate()

print("Spark UI:", spark.sparkContext.uiWebUrl)
print("Default parallelism (cores seen by the driver):", spark.sparkContext.defaultParallelism)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/03 18:07:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark UI: http://1240298c08c4:4040
Default parallelism (cores seen by the driver): 2


In [3]:
import sys
sys.path.append("/workspace/apps")

from spark_profiler import SparkProfiler
profiler = SparkProfiler(spark)



In [4]:
trips = spark.read.parquet(dataFolder + "yellow_tripdata_2020-01.parquet")
zones = spark.read.csv(dataFolder + "taxi_zone_lookup.csv", header=True, inferSchema=True)

## Integração do M1
### Data ingestion and cleaning
Load the monthly Parquet files. Enforce the schema
(cast types, handle nulls)

In [5]:
from pyspark.sql import functions as F
with (((profiler.profile("1-ingestion-cleaning")))):

    trips_typed = (trips
        .withColumn("tpep_pickup_datetime",  F.col("tpep_pickup_datetime").cast("timestamp"))
        .withColumn("tpep_dropoff_datetime", F.col("tpep_dropoff_datetime").cast("timestamp"))
        .withColumn("passenger_count", F.col("passenger_count").cast("int"))
        .withColumn("trip_distance",   F.col("trip_distance").cast("double"))
        .withColumn("fare_amount",     F.col("fare_amount").cast("double"))
        .withColumn("PULocationID",    F.col("PULocationID").cast("int"))
        .withColumn("DOLocationID",    F.col("DOLocationID").cast("int"))
        .withColumn("year", F.year("tpep_pickup_datetime"))
    )
    trips_typed = (trips_typed
                    .filter("fare_amount > 0")
                    .filter("trip_distance > 0")
                    .filter("passenger_count > 0 ")
                    .filter("passenger_count < 6")
                    .filter("tpep_dropoff_datetime > tpep_pickup_datetime")
                    .filter("PULocationID BETWEEN 1 AND 263")
                    .filter("DOLocationID BETWEEN 1 AND 263"))
    print(trips.count())
    print(trips_typed.count())


6405008
5955718
[1-ingestion-cleaning] wall time: 4.58s (6 stages, 26 tasks) - see profiler.report() / profiler.compare(...)


# M2

In [10]:
with profiler.profile("2-temporal-demand-analysis"):
    trips2 = (trips_typed
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime"))
        .withColumn("duration_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60))

    joined = (trips2
        .join(zones, trips2.PULocationID == zones.LocationID, "left")
        .withColumnRenamed("Borough", "pickup_borough"))

    m2 = (joined
        .groupBy("pickup_borough", "day_of_week", "hour_of_day")
        .agg(
            F.count("*").alias("trip_count"),
            F.avg("fare_amount").alias("avg_fare"),
            F.avg("duration_min").alias("avg_duration_min"),
            F.avg("trip_distance").alias("avg_distance"),
        ))

    m2.count()

    m2.orderBy("pickup_borough", "day_of_week", "hour_of_day").show(20)

    (m2.write
        .mode("overwrite")
        .partitionBy("pickup_borough")
        .parquet("/data/output/m2_temporal_demand"))



+--------------+-----------+-----------+----------+------------------+------------------+------------------+
|pickup_borough|day_of_week|hour_of_day|trip_count|          avg_fare|  avg_duration_min|      avg_distance|
+--------------+-----------+-----------+----------+------------------+------------------+------------------+
|         Bronx|          1|          0|        30|             14.55|13.391111111111114| 4.120333333333333|
|         Bronx|          1|          1|        20|             13.75|12.404166666666667|3.8004999999999995|
|         Bronx|          1|          2|        12|10.666666666666666| 8.947222222222223|2.8433333333333333|
|         Bronx|          1|          3|        30|15.466666666666667|12.410555555555556| 4.560666666666667|
|         Bronx|          1|          4|        27|10.407407407407407|  8.68827160493827|2.5659259259259266|
|         Bronx|          1|          5|        12|17.208333333333332|10.911111111111111| 3.480833333333333|
|         Bronx|   

# Integração M3
## Zone-level statistics
For each pickup zone and month, compute mean and
standard deviation of fare_amount and trip_distance


In [11]:
trips_typed.printSchema()
zones.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)
 |-- year: integer (nullable = true)

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Z

In [12]:
with (profiler.profile("M3- Zone Level statistics")):
    mean_and_sdev = trips_typed.groupBy("PULocationID")

    mean_and_sdev = (trips_typed
                     .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
        .groupBy("PULocationID", "month")
        .agg(
            F.count("*").alias("trips"),
            F.avg("fare_amount").alias("fare_mean"),
            F.stddev("fare_amount").alias("fare_std"),
            F.avg("trip_distance").alias("dist_mean"),
            F.stddev("trip_distance").alias("dist_std"),
        ))
    # mean_and_sdev.printSchema()
    # nomes_final = mean_and_sdev.join(zones, mean_and_sdev["PULocationID"] == zones["LocationID"])
    # nomes_final.select("Zone", "Month", "fare_mean", "fare_std", "dist_mean", "dist_std").show()


[M3- Zone Level statistics] wall time: 0.05s (0 stages, 0 tasks) - see profiler.report() / profiler.compare(...)
